# 📖 Notebook 4: Reserve Price, Bid Increments & Proxy Bidding

The bid engine from Notebook 1 accepts **any bid higher than the current max**. Real auction sites like eBay do three things on top of that:

1. **Reserve price** — a hidden minimum the seller is willing to accept. If no bid meets it, the item doesn't sell.
2. **Minimum bid increment** — the next bid must beat the current max by at least $X (otherwise people spam $0.01 bumps forever).
3. **Proxy / automatic bidding** — you tell the system your **max willing price**, and it bids on your behalf just enough to stay on top.

These are core auction rules — not optional features. In this notebook we'll layer them on top of the OCC bidder from Notebook 1.

## Learning Objectives

By the end of this notebook, you'll understand:
- How a **reserve price** changes the end-of-auction outcome (sold vs. unsold)
- Why **minimum bid increments** exist and how they're typically scaled by price
- How **proxy bidding** (eBay's classic feature) works under the hood
- How to extend a schema without destructively changing it (add columns, keep existing data)

## 🛠️ Setup

```bash
cd system-designs/online-auction
docker-compose up -d
```

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
from decimal import Decimal

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "auction_demo", "user": "demo", "password": "demo",
}

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

try:
    conn = get_db_connection(); conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker-compose up -d")

---
## 🗃️ Adding Columns Without Breaking Anything

We'll add three new columns to `auctions`:
- `reserve_price` — optional; if NULL, no reserve
- `bid_increment` — minimum step; default $1
- `max_proxy_bid_user_id` / `max_proxy_bid_amount` — tracks the current *auto-bidder* and their ceiling

Using `ALTER TABLE ... ADD COLUMN IF NOT EXISTS` is idempotent — safe to re-run.

> **Why not drop and recreate the table?** In a real system you can't: dropping the table destroys every user's bid history. Additive migrations are the rule of thumb in production.

In [ ]:
conn = get_db_connection()
conn.autocommit = True
cur = conn.cursor()

cur.execute(
    "ALTER TABLE auctions "
    "ADD COLUMN IF NOT EXISTS reserve_price DECIMAL(12,2), "
    "ADD COLUMN IF NOT EXISTS bid_increment DECIMAL(12,2) NOT NULL DEFAULT 1.00, "
    "ADD COLUMN IF NOT EXISTS max_proxy_bid_user_id INTEGER REFERENCES users(id), "
    "ADD COLUMN IF NOT EXISTS max_proxy_bid_amount DECIMAL(12,2)"
)

print("✅ Schema extended (reserve_price, bid_increment, proxy bid tracking)")

cur.execute(
    "SELECT column_name, data_type "
    "FROM information_schema.columns "
    "WHERE table_name = 'auctions' "
    "ORDER BY ordinal_position"
)
for name, dtype in cur.fetchall():
    print(f"   {name:<25} {dtype}")

conn.close()

---
## 💰 Part 1: Reserve Price

A **reserve price** is the minimum the seller is willing to accept. It's set when the auction is created, and it's usually **hidden** from bidders. Two rules:

1. Bids above the current max are still accepted, even if they're below reserve.
2. When the auction ends, the item only **sells** if the highest bid ≥ reserve. Otherwise it's **unsold** and the seller keeps it.

### Why hidden?
If the reserve is visible, bidders just wait for it and offer exactly that — they never compete. Hiding the reserve turns it into a "floor" the market has to naturally reach.

Let's create a reserve-price auction and walk through the end-of-auction logic.

In [ ]:
def create_reserve_auction(seller_id, item_name, starting_price, reserve_price, duration_days=7):
    """Create an auction with a hidden reserve price."""
    if reserve_price < starting_price:
        raise ValueError("Reserve price cannot be below the starting price")

    conn = get_db_connection()
    cur = conn.cursor()
    try:
        cur.execute(
            "INSERT INTO items (seller_id, name) VALUES (%s, %s) RETURNING id",
            (seller_id, item_name),
        )
        item_id = cur.fetchone()[0]

        cur.execute(
            "INSERT INTO auctions ("
            "  item_id, seller_id, starting_price, max_bid_amount, "
            "  reserve_price, end_date, status"
            ") VALUES (%s, %s, %s, %s, %s, NOW() + (%s || ' days')::interval, 'active') "
            "RETURNING id",
            (item_id, seller_id, starting_price, starting_price,
             reserve_price, str(duration_days)),
        )
        auction_id = cur.fetchone()[0]
        conn.commit()
        return auction_id
    finally:
        conn.close()

# Seller lists a rare book: starts at $50, reserve is $500.
auction_id = create_reserve_auction(
    seller_id=1,
    item_name="Rare First-Edition Book",
    starting_price=50.00,
    reserve_price=500.00,
    duration_days=7,
)
print(f"📕 Created reserve auction #{auction_id} — starts at $50, reserve $500 (hidden)")

In [ ]:
def end_auction_with_reserve(auction_id):
    """
    End an auction and compute the outcome:
    - sold    → max bid >= reserve_price (or no reserve), and there was a bidder
    - unsold  → there was a bid but it didn't meet reserve
    - no_bids → nobody bid at all
    """
    conn = get_db_connection()
    cur = conn.cursor()
    try:
        cur.execute(
            "SELECT max_bid_amount, max_bid_user_id, reserve_price, starting_price "
            "FROM auctions WHERE id = %s FOR UPDATE",
            (auction_id,),
        )
        row = cur.fetchone()
        if not row:
            return {"outcome": "not_found"}

        max_bid, max_user, reserve, starting = row
        max_bid = float(max_bid or 0)
        reserve = float(reserve) if reserve is not None else None
        starting = float(starting)

        if max_user is None or max_bid <= starting:
            outcome = "no_bids"
            winner = None
        elif reserve is not None and max_bid < reserve:
            outcome = "unsold_reserve_not_met"
            winner = None
        else:
            outcome = "sold"
            winner = max_user

        cur.execute(
            "UPDATE auctions SET status = 'ended', updated_at = NOW() WHERE id = %s",
            (auction_id,),
        )
        conn.commit()

        return {
            "outcome": outcome,
            "winner_id": winner,
            "final_price": max_bid,
            "reserve_price": reserve,
        }
    finally:
        conn.close()

# Simulate: max bid was $300, reserve was $500 → unsold
conn = get_db_connection(); conn.autocommit = True
cur = conn.cursor()
cur.execute(
    "UPDATE auctions SET max_bid_amount = 300, max_bid_user_id = 5 WHERE id = %s",
    (auction_id,),
)
conn.close()

result = end_auction_with_reserve(auction_id)
print(f"📊 Outcome: {result}")
if result['outcome'] == 'unsold_reserve_not_met':
    print(f"   💡 Highest bid ${result['final_price']:.2f} didn't meet the hidden reserve.")
    print(f"      The seller keeps the item. No transaction happens.")

---
## 📏 Part 2: Minimum Bid Increment

Without a minimum increment, bidders can play games:

```
Max bid: $100.00
User A bids: $100.01  ✅ accepted — now they lead
User B bids: $100.02  ✅ accepted
User A bids: $100.03  ✅ accepted
... forever
```

This wastes everyone's time and floods the system with tiny updates. Every real auction platform uses **bid increments** that grow with price. eBay's ladder, roughly:

| Current price | Minimum increment |
|---|---|
| $0.01–$0.99 | $0.05 |
| $1.00–$4.99 | $0.25 |
| $5.00–$24.99 | $0.50 |
| $25.00–$99.99 | $1.00 |
| $100.00–$249.99 | $2.50 |
| $250.00–$499.99 | $5.00 |
| $500.00–$999.99 | $10.00 |
| $1,000.00–$2,499.99 | $25.00 |
| $2,500.00–$4,999.99 | $50.00 |
| $5,000+ | $100.00 |

We'll implement this table and reject bids that don't meet the required step.

In [ ]:
# eBay-style bid increment ladder (approximate).
INCREMENT_LADDER = [
    (Decimal("1.00"),     Decimal("0.05")),
    (Decimal("5.00"),     Decimal("0.25")),
    (Decimal("25.00"),    Decimal("0.50")),
    (Decimal("100.00"),   Decimal("1.00")),
    (Decimal("250.00"),   Decimal("2.50")),
    (Decimal("500.00"),   Decimal("5.00")),
    (Decimal("1000.00"),  Decimal("10.00")),
    (Decimal("2500.00"),  Decimal("25.00")),
    (Decimal("5000.00"),  Decimal("50.00")),
]
TOP_INCREMENT = Decimal("100.00")

def required_increment(current_max):
    """Return the minimum increment for the next bid."""
    for threshold, step in INCREMENT_LADDER:
        if current_max < threshold:
            return step
    return TOP_INCREMENT

# Quick sanity check
print("Current max → required increment:")
for c in [0, 3, 50, 120, 800, 3000, 50_000]:
    inc = required_increment(Decimal(c))
    print(f"   ${c:>7,} → ${inc}")

In [ ]:
def place_bid_with_rules(auction_id, user_id, amount):
    """
    OCC-based bid placement that enforces the minimum bid increment.
    Reserve price is not checked here — it's only checked at end-of-auction.
    """
    amount = Decimal(str(amount))
    conn = get_db_connection()
    cur = conn.cursor()

    try:
        cur.execute(
            "SELECT max_bid_amount, status FROM auctions WHERE id = %s",
            (auction_id,),
        )
        row = cur.fetchone()
        if not row:
            return {"status": "error", "reason": "Auction not found"}
        current_max, status = Decimal(row[0]), row[1]
        if status != "active":
            return {"status": "rejected", "reason": f"Auction is {status}"}

        required = required_increment(current_max)
        min_next = current_max + required

        if amount < min_next:
            cur.execute(
                "INSERT INTO bids (auction_id, user_id, amount, status) VALUES (%s, %s, %s, 'rejected')",
                (auction_id, user_id, amount),
            )
            conn.commit()
            return {
                "status": "rejected",
                "reason": f"Bid must be at least ${min_next} (current ${current_max} + increment ${required})",
            }

        # OCC update
        cur.execute(
            "UPDATE auctions SET max_bid_amount = %s, max_bid_user_id = %s "
            "WHERE id = %s AND max_bid_amount = %s",
            (amount, user_id, auction_id, current_max),
        )
        if cur.rowcount != 1:
            conn.rollback()
            return {"status": "conflict", "reason": "Someone else bid first, please retry"}

        cur.execute(
            "INSERT INTO bids (auction_id, user_id, amount, status) VALUES (%s, %s, %s, 'accepted')",
            (auction_id, user_id, amount),
        )
        conn.commit()
        return {"status": "accepted", "amount": float(amount)}
    finally:
        conn.close()

# Create a fresh auction at $100 for a clean demo
demo_id = create_reserve_auction(2, "Increment Demo Item", 100.00, 100.00, 5)

# At $100, required increment is $2.50 (from the ladder above),
# so the smallest legal next bid is $102.50.
print("1) Bidder tries $100.25 — a penny bump...")
print("   →", place_bid_with_rules(demo_id, 10, 100.25))

print("\n2) Bidder tries $101.00 — still below the $102.50 minimum...")
print("   →", place_bid_with_rules(demo_id, 10, 101.00))

print("\n3) Bidder tries exactly $102.50 — the minimum legal next bid...")
print("   →", place_bid_with_rules(demo_id, 11, 102.50))

print("\n4) Bidder tries $150.00 — well above min...")
print("   →", place_bid_with_rules(demo_id, 11, 150.00))

print("\n5) At $150 the required increment is still $2.50 → min next is $152.50. Bidder tries $152.00...")
print("   →", place_bid_with_rules(demo_id, 12, 152.00))

---
## 🤖 Part 3: Proxy / Automatic Bidding (eBay-style)

This is the feature that made eBay different from a shouting match. Instead of watching the auction and manually counter-bidding, you tell the system: **"my max is $500 — bid for me as needed, up to that."**

### How it works

Every auction tracks two things:
- `max_bid_amount` — the **visible** current highest bid (what shoppers see)
- `max_proxy_bid_amount` — the **hidden ceiling** of the current leading auto-bidder

When a new bidder comes in with their own ceiling:

1. If their ceiling < the minimum next bid → rejected.
2. If their ceiling ≤ the current leader's ceiling → the leader's proxy **just barely outbids them**: visible max = new bidder's ceiling + one increment (capped at leader's ceiling).
3. If their ceiling > the current leader's ceiling → the new bidder **takes over**: visible max = old leader's ceiling + one increment (capped at new bidder's ceiling).

This is why on eBay you sometimes place a bid and immediately see "You've been outbid" — someone with a higher max set their proxy hours ago.

### Concrete example

Current state: max $100, leader Alice with proxy ceiling $200.

- Bob sets ceiling $150 → Alice's proxy bumps visible max to ~$151 (still leading). Bob is outbid instantly.
- Carla sets ceiling $250 → Carla takes over at ~$201 (Alice's ceiling + $1 increment).

Let's implement it.

In [ ]:
def place_proxy_bid(auction_id, user_id, ceiling):
    """
    Place a proxy bid with a ceiling (max willing price).
    Returns the new visible max_bid_amount and who's currently leading.
    """
    ceiling = Decimal(str(ceiling))
    conn = get_db_connection()
    cur = conn.cursor()
    try:
        # Pessimistic lock — the logic has too many branches for a clean OCC update.
        cur.execute(
            "SELECT max_bid_amount, max_bid_user_id, "
            "       max_proxy_bid_amount, max_proxy_bid_user_id, status "
            "FROM auctions WHERE id = %s FOR UPDATE",
            (auction_id,),
        )
        row = cur.fetchone()
        if not row:
            return {"status": "error", "reason": "Auction not found"}

        visible_max = Decimal(row[0])
        leader_ceiling = Decimal(row[2]) if row[2] is not None else None
        leader_user = row[3]
        status = row[4]

        if status != "active":
            return {"status": "rejected", "reason": f"Auction is {status}"}

        required = required_increment(visible_max)
        min_legal = visible_max + required

        if ceiling < min_legal:
            cur.execute(
                "INSERT INTO bids (auction_id, user_id, amount, status) VALUES (%s, %s, %s, 'rejected')",
                (auction_id, user_id, ceiling),
            )
            conn.commit()
            return {
                "status": "rejected",
                "reason": f"Ceiling ${ceiling} below minimum next bid ${min_legal}",
            }

        if leader_ceiling is None:
            # First proxy bid — bidder leads at the minimum next price.
            new_visible = min_legal
            new_leader = user_id
            new_leader_ceiling = ceiling
            action = "first_proxy"
        elif leader_user == user_id:
            # Same user raising their own ceiling — visible price doesn't move.
            new_visible = visible_max
            new_leader = user_id
            new_leader_ceiling = ceiling
            action = "ceiling_raised"
        elif ceiling <= leader_ceiling:
            # Current leader's auto-bidder beats challenger by one increment,
            # capped at the leader's ceiling.
            new_visible = min(ceiling + required_increment(ceiling), leader_ceiling)
            new_leader = leader_user
            new_leader_ceiling = leader_ceiling
            action = "challenger_outbid"
        else:
            # Challenger takes over — visible = old leader's ceiling + one increment,
            # capped at challenger's ceiling.
            new_visible = min(leader_ceiling + required_increment(leader_ceiling), ceiling)
            new_leader = user_id
            new_leader_ceiling = ceiling
            action = "leader_changed"

        cur.execute(
            "INSERT INTO bids (auction_id, user_id, amount, status) VALUES (%s, %s, %s, 'accepted')",
            (auction_id, new_leader, new_visible),
        )
        cur.execute(
            "UPDATE auctions SET max_bid_amount = %s, max_bid_user_id = %s, "
            "    max_proxy_bid_amount = %s, max_proxy_bid_user_id = %s "
            "WHERE id = %s",
            (new_visible, new_leader, new_leader_ceiling, new_leader, auction_id),
        )
        conn.commit()

        return {
            "status": "accepted",
            "action": action,
            "visible_max": float(new_visible),
            "leader_id": new_leader,
        }
    finally:
        conn.close()

# Fresh auction for the proxy demo
proxy_id = create_reserve_auction(3, "Proxy Demo Painting", 100.00, 100.00, 5)

print(f"🎨 Auction #{proxy_id} — starting at $100")
print()

print("1) Alice sets proxy ceiling of $200 (no other bidders yet)")
print("   →", place_proxy_bid(proxy_id, user_id=10, ceiling=200))

print("\n2) Bob comes in with a $150 ceiling — Alice's proxy should outbid him")
print("   →", place_proxy_bid(proxy_id, user_id=20, ceiling=150))

print("\n3) Carla sets a $250 ceiling — she should take over")
print("   →", place_proxy_bid(proxy_id, user_id=30, ceiling=250))

print("\n4) Alice raises her ceiling to $500 — she takes over again")
print("   →", place_proxy_bid(proxy_id, user_id=10, ceiling=500))

In [ ]:
# Full bid history + current state
conn = get_db_connection()
cur = conn.cursor()

cur.execute(
    "SELECT max_bid_amount, max_bid_user_id, "
    "       max_proxy_bid_amount, max_proxy_bid_user_id "
    "FROM auctions WHERE id = %s",
    (proxy_id,),
)
row = cur.fetchone()
print("📋 Final auction state:")
print(f"   Visible current bid: ${row[0]} by User {row[1]}")
print(f"   Hidden proxy ceiling: ${row[2]} (belongs to User {row[3]})")
print()

cur.execute(
    "SELECT user_id, amount, status FROM bids WHERE auction_id = %s ORDER BY id",
    (proxy_id,),
)
print("📜 Bid log (what shoppers would see on the auction page):")
for uid, amt, status in cur.fetchall():
    print(f"   User {uid:>2}  ${amt:<8}  {status}")
conn.close()

---
## 🧪 Real-World Corner Case: "I got outbid before I saw the page"

Proxy bidding means the visible price can jump the instant someone places a bid — because an existing proxy responded. From the new bidder's perspective it looks like cheating: *"I placed a bid and was immediately outbid by $1!"*

This is working as designed. The UX fix is to **show only the current visible max**, never the proxy ceiling, and to explain proxy bidding in the help text.

A good auction page tells you three things:
1. Current price
2. Minimum next bid
3. A gentle reminder that entering a max bid uses proxy bidding

## 🧠 Summary

| Feature | What it prevents | Where it lives |
|---|---|---|
| Reserve price | Seller being forced to sell too cheap | `auctions.reserve_price`, checked at end-of-auction |
| Bid increment | Penny-bumping spam | Computed from current max at bid time |
| Proxy bidding | Users having to manually watch auctions | Hidden ceiling per-auction, resolved on each new bid |

### In Production
- Reserve prices are almost always **hidden** — the seller may be shown a badge like "Reserve not met" during the auction.
- Increment ladders are a **product decision**; they're driven by market norms, not technology. eBay, Sotheby's, and Christie's all publish theirs.
- Proxy bidding plays nicely with **anti-sniping** (Notebook 2): a last-second bid often just raises an existing proxy ceiling and doesn't change the leader, so the extension only fires when it actually matters.
- All three rules can be enforced on the **server side** only. Never trust the client to respect the increment.

### What You've Learned Across All 4 Notebooks

| Notebook | Core Problem | Solution |
|---|---|---|
| 1. Bid Processing | Race conditions | OCC, row locking, Redis Lua |
| 2. Lifecycle | States and expiration | State machine, idempotent closers, anti-sniping |
| 3. Notifications | Pushing updates | Redis Pub/Sub, SSE, multi-server |
| 4. Advanced Rules | Real auction semantics | Reserve, increments, proxy bidding |